# Híbrido SARIMAX + LSTM (residuos, multi-output)

SARIMAX(2,0,2)(1,0,1,7) captura la estructura lineal/estacional. Un LSTM se entrena sobre los **residuos** de SARIMAX (lo que el modelo lineal no explica). La predicción final es: `SARIMAX + corrección_LSTM`.

**Versión multi-output:** el LSTM de residuos predice el bloque completo de una sola pasada (sin retroalimentación paso a paso), eliminando la acumulación de error que sí afectó a la primera versión recursiva. Arquitectura más profunda: `hidden_size=128`, `num_layers=3` (antes 64/2).

Se entrena **un modelo de residuos distinto por horizonte** (1, 7, 30 días).

Comparación base: SARIMAX solo (MAE 40.30 / 56.65 / 74.53) vs. híbrido recursivo anterior (empeoró: MAE 43.50 / 66.78 / 103.37).

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, median_absolute_error, max_error
import joblib
import warnings
warnings.filterwarnings('ignore')

SEED = 333
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE = Path.cwd().parents[1] / 'data' / 'processed'
df = pd.read_parquet(BASE / 'dataset_consolidado.parquet')
df.index = pd.to_datetime(df.index)

train = df[df.index < '2025-08-01']
test = df[(df.index >= '2025-08-01') & (df.index <= '2026-07-31')]
covariables = ['gen_termica', 'ONI', 'aportes_energia_gwh']

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
VENTANA = 30
print(f"Train: {len(train)} días | Test: {len(test)} días | Device: {device}")

## Paso 1 — Ajustar SARIMAX (idéntico al ya validado)

In [ ]:
modelo_sarimax = SARIMAX(
    train['precio_bolsa'],
    exog=train[covariables],
    order=(2, 0, 2),
    seasonal_order=(1, 0, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
resultado_sarimax = modelo_sarimax.fit(disp=False)
print("SARIMAX ajustado. AIC:", round(resultado_sarimax.aic, 2))

## Paso 2 — Residuos in-sample (lo que SARIMAX no explica en train)

In [ ]:
residuos_train = resultado_sarimax.resid  # ya alineado al índice de train

print(f"Residuos: media={residuos_train.mean():.2f}, std={residuos_train.std():.2f}")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(residuos_train.index, residuos_train.values, color='darkred', linewidth=0.5)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_title('Residuos de SARIMAX en train — ¿hay patrón aprovechable o es ruido?')
ax.set_ylabel('Residuo ($/kWh)')
plt.tight_layout()
plt.show()

In [ ]:
scaler_resid = StandardScaler()
residuos_scaled = scaler_resid.fit_transform(residuos_train.values.reshape(-1, 1)).flatten()
print(f"Residuos escalados: {len(residuos_scaled)} puntos")

## Paso 3 — Arquitectura multi-output (más profunda: 128 unidades, 3 capas)

In [ ]:
class ModeloLSTM_MultiOutput(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=3, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # una salida por cada paso del horizonte


def crear_ventanas_multioutput(data, ventana, horizonte):
    X, y = [], []
    for i in range(len(data) - ventana - horizonte + 1):
        X.append(data[i:i+ventana])
        y.append(data[i+ventana:i+ventana+horizonte])
    return np.array(X), np.array(y)


def entrenar_residual_multioutput(horizonte, residuos_scaled, hidden_size=128, num_layers=3, epochs=50):
    X, y = crear_ventanas_multioutput(residuos_scaled, VENTANA, horizonte)
    X_t = torch.FloatTensor(X).unsqueeze(-1).to(device)
    y_t = torch.FloatTensor(y).to(device)

    loader_h = DataLoader(TensorDataset(X_t, y_t), batch_size=32, shuffle=True)

    modelo = ModeloLSTM_MultiOutput(hidden_size=hidden_size, num_layers=num_layers, output_size=horizonte).to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.001)
    crit = nn.MSELoss()

    for epoch in range(epochs):
        modelo.train()
        perdida_epoch = 0
        for X_batch, y_batch in loader_h:
            opt.zero_grad()
            pred = modelo(X_batch)
            loss = crit(pred, y_batch)
            loss.backward()
            opt.step()
            perdida_epoch += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} — Loss: {perdida_epoch/len(loader_h):.6f}")

    return modelo

## Paso 4 — Walk-forward híbrido multi-output en 1, 7 y 30 días

In [ ]:
def walk_forward_hibrido_multioutput(resultado_sarimax_fit, modelo_residual, train, test, covariables,
                                      residuos_scaled_iniciales, scaler_resid, pasos):
    predicciones_hibrido = []
    predicciones_sarimax_solo = []
    fechas_pred = []
    resultado_actual = resultado_sarimax_fit
    residuos_actuales = list(residuos_scaled_iniciales)

    for i in range(0, len(test), pasos):
        pasos_bloque = min(pasos, len(test) - i)
        exog_bloque = test[covariables].iloc[i:i+pasos_bloque]

        pred_sarimax = resultado_actual.forecast(steps=pasos_bloque, exog=exog_bloque)

        modelo_residual.eval()
        with torch.no_grad():
            contexto = np.array(residuos_actuales[-VENTANA:], dtype=np.float32).reshape(1, VENTANA, 1)
            contexto_t = torch.FloatTensor(contexto).to(device)
            pred_residuo_scaled = modelo_residual(contexto_t).cpu().numpy().flatten()[:pasos_bloque]

        pred_residuo = scaler_resid.inverse_transform(pred_residuo_scaled.reshape(-1, 1)).flatten()
        pred_hibrido = pred_sarimax.values + pred_residuo

        predicciones_hibrido.extend(pred_hibrido)
        predicciones_sarimax_solo.extend(pred_sarimax.values)
        fechas_pred.extend(test.index[i:i+pasos_bloque])

        y_real_bloque = test['precio_bolsa'].iloc[i:i+pasos_bloque]
        residuo_real = y_real_bloque.values - pred_sarimax.values
        residuo_real_scaled = scaler_resid.transform(residuo_real.reshape(-1, 1)).flatten()
        residuos_actuales.extend(residuo_real_scaled.tolist())

        resultado_actual = resultado_actual.append(y_real_bloque, exog=exog_bloque, refit=False)

    return (pd.Series(predicciones_hibrido, index=fechas_pred),
            pd.Series(predicciones_sarimax_solo, index=fechas_pred))

In [ ]:
def encontrar_raiz_repo(marcador='.git'):
    actual = Path.cwd()
    while actual != actual.parent:
        if (actual / marcador).exists():
            return actual
        actual = actual.parent
    raise FileNotFoundError("No se encontró la raíz del repo")


def agregar_metricas(nombre, real, pred, archivo, horizonte):
    real = np.array(real); pred = np.array(pred)
    mae = np.mean(np.abs(real - pred))
    rmse = np.sqrt(np.mean((real - pred)**2))
    mape = np.mean(np.abs((real - pred) / real)) * 100
    smape = np.mean(2 * np.abs(real - pred) / (np.abs(real) + np.abs(pred))) * 100
    r2 = r2_score(real, pred)
    medae = median_absolute_error(real, pred)
    max_err = max_error(real, pred)
    sesgo = np.mean(pred - real)

    nueva_fila = pd.DataFrame([{
        'Modelo': nombre, 'Horizonte': horizonte,
        'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE (%)': round(mape, 2),
        'sMAPE (%)': round(smape, 2), 'R²': round(r2, 4), 'MedAE': round(medae, 2),
        'MaxError': round(max_err, 2), 'Sesgo': round(sesgo, 2)
    }])

    if archivo.exists():
        tabla = pd.read_parquet(archivo)
        tabla = tabla[~((tabla['Modelo'] == nombre) & (tabla['Horizonte'] == horizonte))]
        tabla = pd.concat([tabla, nueva_fila], ignore_index=True)
    else:
        tabla = nueva_fila
    tabla.to_parquet(archivo, index=False)
    print(f"'{nombre}' (horizonte={horizonte}d) guardado en {archivo}")
    return tabla


raiz = encontrar_raiz_repo()
archivo_tabla = raiz / 'data' / 'processed' / 'tabla_metricas.parquet'
print("Archivo de métricas:", archivo_tabla)

In [ ]:
resultados_horizonte = {}
modelos_residuales = {}

for horizonte in [1, 7, 30]:
    print(f"\n{'='*70}\nHorizonte: {horizonte} día(s) — entrenando residual multi-output\n{'='*70}")

    modelo_residual_h = entrenar_residual_multioutput(horizonte, residuos_scaled, hidden_size=128, num_layers=3)
    modelos_residuales[horizonte] = modelo_residual_h

    pred_hibrido, pred_sarimax_solo = walk_forward_hibrido_multioutput(
        resultado_sarimax, modelo_residual_h, train, test, covariables,
        residuos_scaled, scaler_resid, pasos=horizonte
    )
    real = test['precio_bolsa'].loc[pred_hibrido.index]

    mae_hibrido = np.mean(np.abs(real - pred_hibrido))
    mae_sarimax_solo = np.mean(np.abs(real - pred_sarimax_solo))
    rmse_hibrido = np.sqrt(np.mean((real - pred_hibrido)**2))
    mape_hibrido = np.mean(np.abs((real - pred_hibrido) / real)) * 100

    print(f"\nSARIMAX solo:                  MAE={mae_sarimax_solo:.2f}")
    print(f"Híbrido multi-output (128,3):  MAE={mae_hibrido:.2f} | RMSE={rmse_hibrido:.2f} | MAPE={mape_hibrido:.2f}%")
    print(f"{'Mejora' if mae_hibrido < mae_sarimax_solo else 'Empeora'}: {abs(mae_sarimax_solo - mae_hibrido):.2f} ({abs(mae_sarimax_solo - mae_hibrido)/mae_sarimax_solo*100:.1f}%)")

    agregar_metricas('SARIMAX+LSTM Híbrido (multi-output)', real, pred_hibrido, archivo_tabla, horizonte)
    resultados_horizonte[horizonte] = {'hibrido': pred_hibrido, 'sarimax_solo': pred_sarimax_solo, 'real': real}

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for ax, h in zip(axes, [1, 7, 30]):
    r = resultados_horizonte[h]
    ax.plot(r['real'].index, r['real'].values, color='darkblue', linewidth=0.8, linestyle='--', label='Real')
    ax.plot(r['sarimax_solo'].index, r['sarimax_solo'].values, color='green', linewidth=0.8, alpha=0.7, label='SARIMAX solo')
    ax.plot(r['hibrido'].index, r['hibrido'].values, color='red', linewidth=0.9, label='Híbrido multi-output')
    ax.set_ylabel('$/kWh')
    ax.set_title(f'Horizonte: {h} día(s)')
    ax.legend(fontsize=8)

axes[-1].tick_params(axis='x', rotation=90)
fig.suptitle('SARIMAX solo vs. Híbrido SARIMAX+LSTM (multi-output)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
raiz.joinpath('models').mkdir(exist_ok=True)
resultado_sarimax.save(raiz / 'models' / 'sarimax_hibrido_base.pickle')
joblib.dump(scaler_resid, raiz / 'models' / 'scaler_residual.pkl')

for horizonte, modelo in modelos_residuales.items():
    torch.save(modelo.state_dict(), raiz / 'models' / f'lstm_residual_multioutput_h{horizonte}.pt')

print("Modelos híbridos guardados (SARIMAX + 3 LSTM de residuos, uno por horizonte)")